In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

Load Both Files

In [ ]:
#Load Both Files
train = pd.read_csv(r"C:\Users\user\credit-card-fraud-detection\data\fraudTrain.csv")
test = pd.read_csv(r"C:\Users\user\credit-card-fraud-detection\data\fraudTest.csv")

print(train.shape)
print(test.shape)

Combine Into One Dataset

In [ ]:
#Combine Into One Dataset
data = pd.concat([train, test], ignore_index=True)
print(data.shape)
data.head()

In [ ]:
#Column names and data types
data.info()

Basic statistics

In [ ]:
#basic statistics
data.describe()

In [ ]:
#all column names
data.columns.tolist()

 Check for Missing Values

In [ ]:
#check for missing values
data.isnull().sum()


Check for Duplicates

In [ ]:
#check duplicate
data.duplicated().sum()
#if drop duplicates
data = data.drop_duplicates()


 Fix Data Types

In [ ]:
# Convert transaction datetime from string to proper datetime
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])
# Convert date of birth from string to proper datetime
data['dob'] = pd.to_datetime(data['dob'], format='%Y-%m-%d')
#confirm
data.info()


Engineer Useful Columns
These new columns will power your SQL queries and Power BI dashboard later.

In [ ]:
# Extract hour, day, month from transaction time
data['trans_hour']  = data['trans_date_trans_time'].dt.hour
data['trans_day']   = data['trans_date_trans_time'].dt.day_name()
data['trans_month'] = data['trans_date_trans_time'].dt.month_name()
data['trans_year']  = data['trans_date_trans_time'].dt.year

# Calculate customer age at time of transaction
data['age'] = (data['trans_date_trans_time'] - data['dob']).dt.days // 365

# Create age groups
bins  = [0, 18, 30, 40, 50, 60, 70, 120]
labels = ['<18', '18-30', '31-40', '41-50', '51-60', '61-70', '70+']
data['age_group'] = pd.cut(data['age'], bins=bins, labels=labels)

print(data[['trans_hour', 'trans_day', 'age', 'age_group']].head())

Check is_fraud Balance

In [ ]:
fraud_counts = data['is_fraud'].value_counts()
fraud_pct    = data['is_fraud'].value_counts(normalize=True) * 100

print(fraud_counts)
print(fraud_pct.round(2))

# Quick bar chart
fraud_counts.plot(kind='bar', color=['steelblue', 'crimson'], 
                  title='Fraud vs Legitimate Transactions')
plt.xticks([0, 1], ['Legitimate', 'Fraud'], rotation=0)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

Drop Columns You Don't Need

In [ ]:
# These columns add no analytical value
cols_to_drop = ['first', 'last', 'street', 'trans_num', 
                 'unix_time', 'cc_num']

data = data.drop(columns=cols_to_drop)

print("Final shape:", data.shape)
print("Remaining columns:", data.columns.tolist())

 Save the Cleaned File

In [ ]:
# Save the Cleaned File
data.to_csv(r"C:\Users\user\credit-card-fraud-detection\data\fraud_cleaned.csv", index=False)
print("Cleaned data saved to fraud_cleaned.csv")

In [ ]:
# Reload and check
data = pd.read_csv(r"C:\Users\user\credit-card-fraud-detection\data\fraud_cleaned.csv")

# Drop the unnamed index column
data = data.drop(columns=['Unnamed: 0'])

# Resave correctly - index=False prevents this from happening again
data.to_csv('../data/fraud_cleaned.csv', index=False)

# Confirm
print(data.columns.tolist())
print(data.shape)

Fraud Distribution (already done, confirm %)

In [ ]:
fraud_pct = data['is_fraud'].value_counts(normalize=True) * 100
print(f"Legitimate: {fraud_pct[0]:.2f}%")
print(f"Fraud:      {fraud_pct[1]:.2f}%")

Fraud by Category

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

category_fraud = data[data['is_fraud'] == 1]['category'].value_counts()

category_fraud.plot(kind='bar', color='crimson', ax=ax)
ax.set_title('Fraud Transactions by Merchant Category', fontsize=14)
ax.set_xlabel('Category')
ax.set_ylabel('Number of Fraud Cases')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

 Fraud by Hour of Day

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

hour_fraud = data[data['is_fraud'] == 1]['trans_hour'].value_counts().sort_index()

hour_fraud.plot(kind='line', marker='o', color='crimson', ax=ax)
ax.set_title('Fraud Transactions by Hour of Day', fontsize=14)
ax.set_xlabel('Hour (0 = Midnight, 12 = Noon)')
ax.set_ylabel('Number of Fraud Cases')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

 Fraud Amount vs Legitimate Amount

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

data[data['is_fraud'] == 0]['amt'].clip(upper=1000).plot(
    kind='hist', bins=50, alpha=0.6, color='steelblue', 
    label='Legitimate', ax=ax)

data[data['is_fraud'] == 1]['amt'].clip(upper=1000).plot(
    kind='hist', bins=50, alpha=0.6, color='crimson', 
    label='Fraud', ax=ax)

ax.set_title('Transaction Amount: Fraud vs Legitimate', fontsize=14)
ax.set_xlabel('Amount (capped at $1000)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

# Also print average amounts
print("Avg Legitimate Amount: $", round(data[data['is_fraud']==0]['amt'].mean(), 2))
print("Avg Fraud Amount:      $", round(data[data['is_fraud']==1]['amt'].mean(), 2))

 Fraud by Age Group

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

age_fraud = data[data['is_fraud'] == 1]['age_group'].value_counts().sort_index()

age_fraud.plot(kind='bar', color='darkorange', ax=ax)
ax.set_title('Fraud Cases by Age Group', fontsize=14)
ax.set_xlabel('Age Group')
ax.set_ylabel('Number of Fraud Cases')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Fraud by Gender

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

gender_fraud = data[data['is_fraud'] == 1]['gender'].value_counts()

gender_fraud.plot(kind='bar', color=['steelblue', 'hotpink'], ax=ax)
ax.set_title('Fraud Cases by Gender', fontsize=14)
ax.set_xlabel('Gender')
ax.set_ylabel('Number of Fraud Cases')
plt.xticks([0, 1], ['Male', 'Female'], rotation=0)
plt.tight_layout()
plt.show()

Top 10 States by Fraud


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

state_fraud = data[data['is_fraud'] == 1]['state'].value_counts().head(10)

state_fraud.plot(kind='bar', color='purple', ax=ax)
ax.set_title('Top 10 States by Fraud Cases', fontsize=14)
ax.set_xlabel('State')
ax.set_ylabel('Number of Fraud Cases')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Save Your EDA Summary

In [ ]:
# Quick summary stats to reference later
summary = {
    'Total Transactions': len(data),
    'Total Fraud Cases': data['is_fraud'].sum(),
    'Fraud Percentage': f"{data['is_fraud'].mean()*100:.2f}%",
    'Avg Fraud Amount': f"${data[data['is_fraud']==1]['amt'].mean():.2f}",
    'Avg Legit Amount': f"${data[data['is_fraud']==0]['amt'].mean():.2f}",
    'Top Fraud Category': data[data['is_fraud']==1]['category'].value_counts().index[0],
    'Peak Fraud Hour': data[data['is_fraud']==1]['trans_hour'].value_counts().index[0],
    'Top Fraud State': data[data['is_fraud']==1]['state'].value_counts().index[0],
}

for k, v in summary.items():
    print(f"{k:<25}: {v}")

:

#### key findings.


🔴 Fraud is 0.52% — classic imbalance, fraudsters are rare but costly
🔴 Avg fraud amount $530.66 vs legit $67.65 — fraudsters spend 8x more
🔴 Peak hour 22 (10pm) — late night attacks
🔴 Top category grocery_pos — surprising! Usually hidden in plain sight
🔴 Top state NY — high population, high fraud

